In [ ]:
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

from tqdm import tqdm


# Tensorflow: check available devices
all_devices = tf.config.list_physical_devices()
print('Found {} devices: {}'.format( len(all_devices), all_devices ))

---

# Fetch the Dataset

* ESC-50: a dataset for Environmental Sound Classification (https://github.com/karolpiczak/ESC-50, https://www.karolpiczak.com/papers/Piczak2015-ESC-Dataset.pdf)
 * 50 classes, 40 files per class, 5s clips

In [ ]:
# !wget https://github.com/karolpiczak/ESC-50/archive/master.zip
# !unzip master.zip

In [ ]:
fn_csv = 'ESC-50-master/meta/esc50.csv'

df = pd.read_csv(fn_csv)  # reads csv, creates dataframe
print(df.head())
unique_classes = df['category'].unique()  # retrieves a list of unique classes
print(f'\nUnique classes: {unique_classes}\nCount: {len(unique_classes)}')

---

# ESC-5: Curation

Let's select 5 classes (*our_classes*) from ESC-50 to make things a bit faster.

In [ ]:
fn_csv = "ESC-50-master/meta/esc50.csv"

our_classes = ["crying_baby", "dog", "rain", "rooster", "sneezing"]  # Note: This is also our class map for later.
esc5_X = []  # File list
esc5_y = []  # Class list

df = pd.read_csv(fn_csv)  # reads csv, creates dataframe
df.values

for row in df.values:  # (This particular algorithm here works, but is not ideal)
  if any(row[3] == k for k in our_classes):
    esc5_X.append(row[0])  # Filename column
    esc5_y.append(row[3])  # Class column

print( list(zip(esc5_X[:5], esc5_y[:5])) )
print(f"Lengths: esc5_X: {len(esc5_X)}, esc5_y: {len(esc5_y)}")


---

# Splitting the dataset into *train* and *test* subsets

***Tasks:***
* ESC-5 is almost ready. With **sklearn**, split the dataset into **train** and **test** subsets with a 80%/20% split ratio and random state 1337.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(esc5_X, esc5_y, test_size=0.2, random_state=1337)  # Look at your imports for a hint which function to use.
print(f"X: {len(X_train)}, {len(X_test)}; y: {len(y_train)}, {len(y_test)}")

---
# ESC-5: Create mel spectrograms

In [ ]:
### START CODING HERE ###
def extract_mel_spec(X_train, y_train):
  X = []  # feature tensor
  y = []  # target tensor

  mel_bands = 128
  for i, filename in tqdm(enumerate(X_train)):  # tqdm displays a progress bar.
    wav_data, sr = librosa.load(f"ESC-50-master/audio/{filename}",sr=None)  # Use the wav file's sample rate.

    # Features
    mel_spec = librosa.feature.melspectrogram(y=wav_data, sr=sr, n_mels=mel_bands)  # Create mel spectrogram. Output shape: (128, 216) (n_mels, frames)
    mel_spec = mel_spec/mel_bands  # Normalization
    mel_spec = mel_spec.T  # Transposition. Output shape: (216, 128)
    X.append( mel_spec )  # Append to feature tensor

    # Targets == class_name
    targets = np.ones( mel_spec.shape[0] )  # Create a PLACEHOLDER target vector. Output shape: (216) (Note: silent frames are NOT going to be labeled as "silent")
    targets = targets * our_classes.index( y_train[i] )  # Convert values to actual class-index from 'our_classes'
    y.append(targets)  # Append to target tensor

  # Stack tensors
  X = np.vstack(X)
  y = np.hstack(y)

  # Return the tensors
  return X, y


# Call the function on our previously generated lists
X_train_ready, y_train_ready = extract_mel_spec(X_train, y_train)
X_test_ready, y_test_ready = extract_mel_spec(X_test, y_test)
### END CODING HERE ###


print(f"\nShapes: X_train_ready: {X_train_ready.shape}, y_train_ready: {y_train_ready.shape}")
print(f"Shapes: X_test_ready: {X_test_ready.shape}, y_test_ready: {y_test_ready.shape}")

---

# Train a nearest neighbor classifier

***Tasks:***
* Use the features and targets from above to **train/fit** a kNN-classifier from sklearn, with 5 neighbors and uniform weighting.
* Print the scores on the train set and test set. Round to 4 decimals with **np.round**. (This will take some time!)


In [ ]:
# Feature scaling/standardization: removes mean & scales to unit variance
print("Scaling...")
scaler = StandardScaler()
scaler.fit(X_train_ready)
X_train_ready = scaler.transform(X_train_ready)  # Normalize the features here for both train and test
X_test_ready = scaler.transform(X_test_ready)

print("Fitting...")
model = KNeighborsClassifier(n_neighbors=5, weights='uniform')  # Call the kNN-classifier. Look at your imports again for a hint.
model.fit(X_train_ready, y_train_ready)  # Fit/Train the classifier using our generated tensors.

print("Evaluating...")
print(f"Train score: {np.round(model.score(X_train_ready, y_train_ready), decimals=4)}")
print(f"Test score: {np.round(model.score(X_test_ready, y_test_ready), decimals=4)}")

---

# ESC-5: Plot the confusion matrix
We want to learn more about our classifier. How well does it perform per class?

***Tasks:***
* Using sklearn, create a confusion matrix of our classifier over the **test set**
* Normalize the rows, use **our_classes** as axes tick values
* Display the plot

In [ ]:
y_test_pred = model.predict(X_test_ready)  # Predict the test set
cm = confusion_matrix(y_test_pred, y_test_ready, normalize='true')  # Call the confusion matrix function. Look at your imports again for a hint.
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=our_classes)  # Create a display object for the confusion matrix. Look at your imports again for a hint.
disp.plot()
plt.xticks(ticks=np.arange(5), labels=our_classes)  # hint: np.arange(5) = (0, 1, 2, 3, 4)
plt.yticks(ticks=np.arange(5), labels=our_classes)
plt.show()

---

# **Getting to know Tensorflow for Neural Networks**
# Import Tensorflow modules

In [ ]:
from keras.models import Sequential, Model
from keras.layers import Input, Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from keras.optimizers import SGD
from keras.losses import CategoricalCrossentropy, SparseCategoricalCrossentropy
from keras.utils import to_categorical
from keras import regularizers

---

# 1. One-Hot Encoding (OHE) of targets (binary class matrix)

Situation in our dataset:
* We have 5 classes, and only 1 class is always active --> This is a **multi-class** problem.
* Thus, we **categorize** our targets. A OHE'd target with index "3" becomes [0 0 0 1 0] (mapped against the output).
* https://www.tensorflow.org/api_docs/python/tf/keras/utils/to_categorical

---
# 2. Build a simple DNN (Sequential API & Functional API)

* Building an NN can be straightforward. Let's build a simple, extendable
architecture.
* Training and evaluating an NN is very similar to what you saw for the kNN-classifier.

* Sequential API: https://machinelearningmastery.com/tutorial-first-neural-network-python-keras/

* Functional API: https://machinelearningmastery.com/keras-functional-api-deep-learning/

* Model compile and summary 

Now, **train** the model on the **train set** with the **targets** for 10 epochs.


Let's print the training loss and the accuracy

Now, **evaluate** the model on the **test set** and print the scores.

Finally, we would like to see the confusion matrix for this model.
* Use the **predict**-function to classify the **test set**. This will give us "predicted targets".
* Because the targets & predictions are vectors (remember OHE), we need their **argmax** along a certain axis.
 * argmax retrieves the index at which the array contains the highest value.

---
# 3. Build a simple CNN (Functional API)

**CNN: Preparation of tensors**
* For 2D convolutional layers, we need to reshape our tensors.
* 2D conv layers take a 4D-input of the form NHWC: *(num_examples, height, width, channels)*.
  * **num_examples**: (in this case) total number of frames (for the train set)
  * **height**: number of time frames per example, here: 1
  * **width**: feature dimension, here: 128
  * **channels**: audio channels, here: 1 (mono)

**IMPORTANT NOTES:**
* Above numbers are valid for our frame-based approach.
We could also translate *num_examples* to number of files, then each example would have a *height* of 216 (because each file has 216 frames).
* In our particular example, we could also use 1D conv layers with, e.g., kernel_size=(3). Same for MaxPooling.

* Adapt the shape to fit the CNN

**Train** the CNN on the correct tensors for 10 epochs with a batch size of 16.

* Display the loss and the accuracy curves

**Evaluate** the CNN on the **test set** and print the scores.


**Confusion matrix**: Get the predictions for the **test set**, then consider the **argmax** again.

# Repeat at signal level

* Normalize the features here for both train and test


* Adjust the dimensions to fit the 2D CNN

* Create the CNN model and compile it

* Fit it

* loss and accuracy curves

* Evaluate the model

* Predict and confussion matrix

---

# Hyper-parameter tuning

Try playing around with these values and see how it affects the results:

* **learning rate**: go in steps of an order of (half) magnitude, e.g. 1.0, 0.1, 0.01, 0.005...
* **batch size**: e.g. 1, 16, 32, 216. 216 will result in 160 batches. Where have you seen this number before?